# `generate_fixed_step_grid()`

The grid utility `nematics3d.generate_fixed_step_grid()` constructs a two-dimensional regular grid from requested physical extents and fixed step lengths. It is useful when spacing is the primary requirement and the exact number of grid points should be determined automatically.

Three facts are important from the beginning:

- the requested sizes are snapped down to extents compatible with complete fixed steps;
- `alignment="bottom-left"` and `alignment="center"` describe different placements of the same integer grid topology;
- the function returns both continuous two-dimensional coordinates and an integer index grid, because high-level plane-grid code uses the integer topology separately when mapping samples into three-dimensional physical space.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.


In [ ]:
import numpy as np
import nematics3d as n3d


## Minimal example: fixed spacing from the bottom-left corner

The requested sizes below are 5.2 and 3.1, while the spacings are 2 and 1. Only complete steps are retained, so the realized extents are 4 and 3.


In [ ]:
grid, grid_int, size_eff = n3d.generate_fixed_step_grid(
    5.2, 3.1, 2.0, 1.0
)
print("grid shape:", grid.shape)
print("last coordinate:", grid[-1, -1])
print("last integer index:", grid_int[-1, -1])
print("effective size:", size_eff)


## Inputs and outputs

The public signature is:

```python
generate_fixed_step_grid(
    size1,
    size2,
    step1,
    step2,
    alignment="bottom-left",
)
```

### Sizes and step lengths

`size1` and `size2` are finite, non-negative real numbers. They specify the requested coordinate extents along the two grid axes. Zero is valid and produces a single sample along that axis.

`step1` and `step2` are finite real numbers in the inclusive range `[1e-12, np.inf]`. This lower bound prevents zero or numerically meaningless spacings from creating an invalid or impractically large grid. Boolean values, `NaN`, infinity where not permitted, complex values, and non-numeric input are rejected by the shared `Nematics3D` number validator.

### `alignment`

`alignment` accepts exactly two strings:

| Value | Meaning |
| --- | --- |
| `"bottom-left"` | Index `(0, 0)` is placed at coordinate `(0, 0)` and coordinates grow in the positive directions. |
| `"center"` | The grid contains coordinate `(0, 0)` at its central sample and extends symmetrically in both directions. |

### Returned values

The function returns `(grid, grid_int, size_eff)`.

| Return value | Shape / type | Meaning |
| --- | --- | --- |
| `grid` | `(n1, n2, 2)` floating `NumPy` array | Continuous two-dimensional coordinates. |
| `grid_int` | `(n1, n2, 2)` integer `NumPy` array | Discrete grid topology; `grid_int[i, j] == (i, j)`. |
| `size_eff` | two floats | Extents actually covered by the generated grid. |

The integer topology is independent of the chosen physical step lengths. High-level objects such as `PlaneGrid` can therefore use `grid_int` as a sampling layout and map it into a three-dimensional plane using their own physical basis vectors.


## Examples


### Bottom-left alignment

For bottom-left alignment, the number of samples along one axis is

$$n=\left\lfloor\frac{L}{\Delta}\right\rfloor+1,$$

where $L$ is the requested size and $\Delta$ is the step length. The effective size is $(n-1)\Delta$.


In [ ]:
grid, grid_int, size_eff = n3d.generate_fixed_step_grid(
    2.5, 2.0, 1.0, 0.75, alignment="bottom-left"
)
print("coordinates:\n", grid)
print("integer topology:\n", grid_int)
print("effective size:", size_eff)


### Center alignment

Center alignment retains an equal number of complete steps on both sides of zero. The number of samples is therefore always odd, and `(0, 0)` is an actual grid point rather than a geometric point between samples.


In [ ]:
grid_center, grid_int_center, size_eff_center = n3d.generate_fixed_step_grid(
    5.2, 4.4, 1.0, 1.0, alignment="center"
)
center = tuple(np.array(grid_center.shape[:2]) // 2)
print("shape:", grid_center.shape)
print("first coordinate:", grid_center[0, 0])
print("center coordinate:", grid_center[center])
print("last coordinate:", grid_center[-1, -1])
print("effective size:", size_eff_center)


### Zero requested size

A zero extent does not produce an empty grid. It produces one sample at coordinate zero along that axis, with effective size zero.


In [ ]:
grid_zero, grid_int_zero, size_eff_zero = n3d.generate_fixed_step_grid(
    0.0, 0.0, 2.0, 3.0
)
print(grid_zero)
print(grid_int_zero)
print(size_eff_zero)


### Requested size versus effective size

The function never stretches the spacing to fit the requested extent. Instead, it preserves `step1` and `step2` exactly and reports the largest compatible extent not exceeding the request. This distinction matters when downstream geometry should retain a known physical sampling distance.


In [ ]:
for requested in [4.0, 4.4, 4.9]:
    _, _, realized = n3d.generate_fixed_step_grid(
        requested, 1.0, 1.5, 1.0
    )
    print(f"requested={requested:.1f}, realized={realized[0]:.1f}")


## Details

### Bottom-left and center use different snapping rules

Bottom-left alignment retains every complete positive step from zero up to the requested size. Center alignment instead retains only complete **pairs** of steps around zero, so its effective size can be smaller than the bottom-left effective size for the same requested size and spacing.

For center alignment, if $m=\lfloor L/(2\Delta)\rfloor$, then the grid has $2m+1$ samples and effective size $2m\Delta$.

### Why return `grid_int`?

The continuous two-dimensional coordinates are useful when the generated grid is used directly. The integer grid serves a different role: it records only the discrete sampling topology. `PlaneGrid` uses that topology and separately combines it with physical in-plane basis vectors, step lengths, and an origin to construct three-dimensional sample positions.


## Possible issues

### Assuming the requested size is always realized exactly

If the requested size is not compatible with the fixed step, the returned `size_eff` is smaller. Use `size_eff`, not the original request, when later code needs the actual generated extent.

### Expecting center alignment to preserve the same point count

Center alignment enforces symmetry and requires an odd number of samples. It may therefore discard an additional incomplete pair of edge steps compared with bottom-left alignment.

### Very small step lengths

The minimum accepted step is `1e-12`, but a mathematically valid step can still imply an impractically large grid for a macroscopic requested size. Estimate the expected point count before generating very fine grids.


## Implementation notes

**This section is intended for developers. Regular users can safely skip it.**

The implementation constructs the integer topology once with `numpy.indices()`. The floating coordinate array is derived from that topology by converting it to floating point, applying the alignment shift, and multiplying by the requested step lengths. This avoids building separate `numpy.meshgrid()` intermediates for the integer and floating representations. Input numbers and the alignment string are validated through the shared `Nematics3D` datatype helpers rather than through function-specific parsing logic.
